In [1]:
import os
import re
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

# 환경 설정, 모델 및 Processor 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "Salesforce/instructblip-flan-t5-xl"
save_dir = "./models/instructblip"
os.makedirs(save_dir, exist_ok=True)

processor = InstructBlipProcessor.from_pretrained(model_name)
model = InstructBlipForConditionalGeneration.from_pretrained(model_name)

processor.save_pretrained(save_dir)
model.save_pretrained(save_dir)
print(f" 가중치 저장 완료: {save_dir}")

# Prompt 생성 및 정답 추출 함수
def build_prompts(q, A, B, C, D):
    return [
        f"{q}\nA. {A}\nB. {B}\nC. {C}\nD. {D}\nAnswer:",
        f"Question: {q}\nOptions:\n(A) {A}\n(B) {B}\n(C) {C}\n(D) {D}\nPlease select A, B, C, or D.\nAnswer:",
        f"Choose the correct answer (A, B, C, or D):\n{q}\nA: {A}\nB: {B}\nC: {C}\nD: {D}\nYour answer:"
    ]

def extract_answer(text):
    match = re.search(r"\b([ABCD])\b", text.upper())
    return match.group(1) if match else "A"

def move_to_device(batch, device):
    if isinstance(batch, dict):
        return {k: move_to_device(v, device) for k, v in batch.items()}
    elif isinstance(batch, list):
        return [move_to_device(v, device) for v in batch]
    elif isinstance(batch, torch.Tensor):
        return batch.to(device)
    return batch

# 학습 데이터 (Accuracy 확인용)
train_df = pd.read_csv("./open/train.csv")
train_img_dir = "./open/train_input_images"

train_preds = []

for row in tqdm(train_df.itertuples(), total=len(train_df), desc="Train Inference"):
    image_path = os.path.join(train_img_dir, os.path.basename(row.img_path))
    image = Image.open(image_path).convert("RGB")

    votes = []
    prompts = build_prompts(row.Question, row.A, row.B, row.C, row.D)

    for prompt in prompts:
        inputs = processor(images=image, text=prompt, return_tensors="pt")
        inputs = move_to_device(inputs, device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                temperature=0.0
            )

        answer_text = processor.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        votes.append(extract_answer(answer_text))

    final_pred = max(set(votes), key=votes.count)
    train_preds.append(final_pred)

# 정확도 계산
train_acc = (pd.Series(train_preds) == train_df["answer"]).mean()
print(f"Prompt Ensemble Accuracy (Train): {train_acc:.4f}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 가중치 저장 완료: ./models/instructblip


Train Inference:   0%|          | 0/60 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Train Inference:   2%|▏         | 1/60 [00:07<07:48,  7.95s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Train Inference:   3%|▎         | 2/60 [00:15<07:20,  7.60s/it]The following g

Prompt Ensemble Accuracy (Train): 0.8500


In [2]:
# 테스트 데이터 추론 및 제출 파일 생성
test_df = pd.read_csv("./open/test.csv")
sample_sub = pd.read_csv("./open/sample_submission.csv")
test_img_dir = "./open/test_input_images"
test_preds = []

for row in tqdm(test_df.itertuples(), total=len(test_df), desc="Test Inference"):
    image_path = os.path.join(test_img_dir, os.path.basename(row.img_path))
    image = Image.open(image_path).convert("RGB")

    votes = []
    prompts = build_prompts(row.Question, row.A, row.B, row.C, row.D)

    for prompt in prompts:
        inputs = processor(images=image, text=prompt, return_tensors="pt")
        inputs = move_to_device(inputs, device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                temperature=0.0
            )

        answer_text = processor.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        votes.append(extract_answer(answer_text))

    final_pred = max(set(votes), key=votes.count)
    test_preds.append(final_pred)


Test Inference:   0%|          | 0/852 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Test Inference:   0%|          | 1/852 [00:07<1:43:39,  7.31s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Test Inference:   0%|          | 2/852 [00:14<1:41:14,  7.15s/it]The followi

In [4]:
# 최종 제출 파일 저장
os.makedirs("./sub", exist_ok=True)
submission = sample_sub.copy()
submission["answer"] = test_preds
submission.to_csv("./sub/submission.csv", index=False)
print("end")

end
